In [0]:
from pyspark.sql.functions import col, lower, to_timestamp, when

In [0]:
df = spark.table("ecommerce_dev.bronze.orders_raw")

orders_clean = df.dropDuplicates(["order_id"]) \
    .filter(col("order_id").isNotNull()) \
    .withColumn("order_purchase_timestamp", to_timestamp("order_purchase_timestamp")) \
    .withColumn("order_approved_at", to_timestamp("order_approved_at")) \
    .withColumn("order_delivered_carrier_date", to_timestamp("order_delivered_carrier_date")) \
    .withColumn("order_delivered_customer_date", to_timestamp("order_delivered_customer_date")) \
    .withColumn("order_estimated_delivery_date", to_timestamp("order_estimated_delivery_date")) \
    .filter(col("order_status").isNotNull())

orders_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dev.silver.orders_clean")

In [0]:
df = spark.table("ecommerce_dev.bronze.customers_raw")

customers_clean = df.dropDuplicates(["customer_id"]) \
    .filter(col("customer_id").isNotNull()) \
    .withColumn("customer_city", lower(col("customer_city"))) \
    .withColumn("customer_state", lower(col("customer_state")))

customers_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dev.silver.customers_clean")

In [0]:
df = spark.table("ecommerce_dev.bronze.products_raw")

products_clean = df.dropDuplicates(["product_id"]) \
    .fillna({"product_category_name": "unknown"}) \
    .withColumn("product_name_lenght", col("product_name_length").cast("int")) \
    .withColumn("product_description_length", col("product_description_length").cast("int")) \
    .withColumn("product_photos_qty", col("product_photos_qty").cast("int")) \
    .withColumn("product_weight_g", col("product_weight_g").cast("int")) \
    .withColumn("product_length_cm", col("product_length_cm").cast("int")) \
    .withColumn("product_height_cm", col("product_height_cm").cast("int")) \
    .withColumn("product_width_cm", col("product_width_cm").cast("int"))

products_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dev.silver.products_clean")

In [0]:
df = spark.table("ecommerce_dev.bronze.sellers_raw")

sellers_clean = df.dropDuplicates(["seller_id"]) \
    .filter(col("seller_id").isNotNull()) \
    .withColumn("seller_city", lower(col("seller_city"))) \
    .withColumn("seller_state", lower(col("seller_state")))

sellers_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dev.silver.sellers_clean")

In [0]:
df = spark.table("ecommerce_dev.bronze.payments_raw")

payments_clean = df.dropDuplicates() \
    .filter(col("order_id").isNotNull()) \
    .withColumn("payment_type", lower(col("payment_type"))) \
    .withColumn("payment_installments", col("payment_installments").cast("int")) \
    .withColumn("payment_value", col("payment_value").cast("double")) \
    .filter(col("payment_value") > 0)

payments_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dev.silver.payments_clean")

In [0]:
df = spark.table("ecommerce_dev.bronze.reviews_raw")

reviews_clean = df.dropDuplicates(["review_id"]) \
    .withColumn("review_score", col("review_score").cast("int")) \
    .withColumn("review_score", when(col("review_score").isNull(), 0).otherwise(col("review_score")))

reviews_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dev.silver.reviews_clean")

In [0]:
df = spark.table("ecommerce_dev.bronze.order_items_raw")

items_clean = df.dropDuplicates() \
    .filter(col("order_id").isNotNull()) \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double"))

items_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dev.silver.order_items_clean")

In [0]:
df = spark.table("ecommerce_dev.bronze.geolocation_raw")

geo_clean = df.dropDuplicates() \
    .withColumn("geolocation_city", lower(col("geolocation_city"))) \
    .withColumn("geolocation_state", lower(col("geolocation_state")))

geo_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dev.silver.geolocation_clean")

In [0]:
df = spark.table("ecommerce_dev.bronze.product_category_name_translation_raw")

category_clean = df.dropDuplicates() \
    .withColumn("product_category_name", lower(col("product_category_name"))) \
    .withColumn("product_category_name_english", lower(col("product_category_name_english")))

category_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("ecommerce_dev.silver.category_translation_clean")